In [2]:
import os
import re
from pathlib import Path
from semevalpolar.finetuning.instruct.predict import generate_predictions_jsonl
from typing import List

from semevalpolar.utils import get_project_root

In [3]:
data_path = Path("data/splits/val.jsonl")
with data_path.open("r", encoding="utf-8") as f:
	records = f.readlines()

In [4]:
type(records)

list

In [6]:
import re
import codecs
from typing import List

INPUT_RE = re.compile(
    r'Input:\s*(.*?)\s*Reasoning:',
    re.DOTALL
)

def get_all_inputs(records: List[str]) -> List[str]:
    cleaned: List[str] = []

    for text in records:
        m = INPUT_RE.search(text)
        if not m:
            cleaned.append("")
            continue

        # decode escaped sequences like \\n, \\t
        decoded = codecs.decode(m.group(1), "unicode_escape")

        # normalize all whitespace
        normalized = " ".join(decoded.split())

        cleaned.append(normalized)

    return cleaned


In [7]:
dataset = get_all_inputs(records)

In [9]:
generate_predictions_jsonl(dataset)

Generating predictions for 322 texts...


AcceleratorError: CUDA error: out of memory
Search for `cudaErrorMemoryAllocation' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [10]:
import torch

In [11]:
print(torch.cuda.is_available())

True


In [14]:
import re
import json

def extract_labels_jsonl(input_path, output_path):
    pattern = re.compile(r'Final Answer \(output ONLY 0 or 1\):\s*(\d+)')

    with open(input_path, "r", encoding="utf-8") as fin, \
         open(output_path, "w", encoding="utf-8") as fout:

        for line in fin:
            obj = json.loads(line)

            # Prefer extracted_label if it already exists
            if "extracted_label" in obj:
                label = int(obj["extracted_label"])
            else:
                # Fallback: extract from prediction text
                text = obj.get("prediction", "")
                match = pattern.search(text)
                if match is None:
                    continue  # or raise an error if required
                label = int(match.group(1))

            fout.write(json.dumps({"label": label}) + "\n")

In [15]:
extract_labels_jsonl(os.path.join(get_project_root(), "src", "semevalpolar",
                                  "finetuning", "instruct", "predictions", "predictions.jsonl"),
                     os.path.join(get_project_root(), "src", "semevalpolar",
                                  "finetuning", "instruct", "predictions", "gold.jsonl"))

# 1 Example

In [ ]:
def predict(text):
	prompt = f"""Input:
            {text}

            Reasoning:

            """